In [1]:
import pandas as pd
import requests
import json
import re
from pathlib import Path
from datetime import date

GPU_DIR = Path("gpu_dataset_v2")
GPU_DIR.mkdir(exist_ok=True)

COLLECTION_DATE = date.today().isoformat()

TARGET_GPUS = [
    "A100",
    "H100",
    "H200",
    "L4",
    "T4",
    "V100"
]

TARGET_REGIONS = {
    "AWS": ["us-east-1"],
    "Azure": ["eastus"],
    "GCP": ["us-central1"]
}

print("GPU dataset folder ready:", GPU_DIR.absolute())
print("Collection date:", COLLECTION_DATE)

GPU dataset folder ready: C:\Users\user\Cloud Project\gpu_dataset_v2
Collection date: 2026-06-05


In [2]:
def fetch_azure_retail_prices(filter_query, max_pages=20):
    base_url = "https://prices.azure.com/api/retail/prices"
    params = {
        "$filter": filter_query
    }

    rows = []
    url = base_url
    page = 0

    while url and page < max_pages:
        if page == 0:
            r = requests.get(url, params=params, timeout=60)
        else:
            r = requests.get(url, timeout=60)

        r.raise_for_status()
        data = r.json()

        rows.extend(data.get("Items", []))
        url = data.get("NextPageLink")
        page += 1

        print("Page", page, "rows:", len(rows))

    return pd.DataFrame(rows)

In [3]:
azure_filter = (
    "serviceName eq 'Virtual Machines' "
    "and armRegionName eq 'eastus' "
    "and priceType eq 'Consumption'"
)

azure_raw = fetch_azure_retail_prices(azure_filter, max_pages=30)

print(azure_raw.shape)
azure_raw.head()

Page 1 rows: 1000
Page 2 rows: 2000
Page 3 rows: 3000
Page 4 rows: 4000
Page 5 rows: 5000
Page 6 rows: 6000
Page 7 rows: 7000
Page 8 rows: 8000
Page 9 rows: 8537
(8537, 21)


,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,...,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,effectiveEndDate
0,USD,0.0,6.2616,6.2616,eastus,US East,2026-04-01T00:00:00Z,000a689e-f5cc-55f2-bb67-f022dd5ceeb1,NC320dsxlRTX6Kv6 Spot,DZH318Z0R67S,...,Virtual Machines NCdsxlRTX6kv6 Windows,NC320dsxlRTX6Kv6 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_NC320ds_xl_RTXPRO6000BSE_v6,NaN
1,USD,0.0,0.0688,0.0688,eastus,US East,2023-07-01T00:00:00Z,000c494f-505a-508d-84e3-6c512039061f,DC8as v5 Low Priority,DZH318Z09B6C,...,DCasv5-series Linux,Standard_DC8as_v5 Low Priority,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_DC8as_v5,NaN
2,USD,0.0,4.6790,4.6790,eastus,US East,2024-08-01T00:00:00Z,00114c3c-893a-5141-8cf2-7582013ce76f,FX64mds v2 Low Priority,DZH318Z0H7DN,...,Virtual Machines FXmdsv2 Series Windows,FX64mds v2 Low Priority,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_FX64mds_v2,NaN
3,USD,0.0,1.6140,1.6140,eastus,US East,2022-01-01T00:00:00Z,0029e038-4955-5ad0-b930-ae3c68914557,E112iads v5 Low Priority,DZH318Z093X0,...,Virtual Machines Eadsv5 Series,E112iads v5 Low Priority,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_E112iads_v5,NaN
4,USD,0.0,6.6940,6.6940,eastus,US East,2023-06-01T00:00:00Z,0036165e-c869-5d05-b9a3-4d7885d96504,HX176-48rs Low Priority,DZH318Z0K1LG,...,Virtual Machines HXrs Windows,HX176-48rs Low Priority,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_HX176-48rs,NaN


In [4]:
azure_gpu_keywords = [
    "A100",
    "H100",
    "H200",
    "L4",
    "T4",
    "V100",
    "NC",
    "ND",
    "NV"
]

pattern = "|".join(azure_gpu_keywords)

azure_gpu = azure_raw[
    azure_raw["productName"].str.contains(pattern, case=False, na=False) |
    azure_raw["skuName"].str.contains(pattern, case=False, na=False) |
    azure_raw["meterName"].str.contains(pattern, case=False, na=False)
].copy()

azure_gpu[[
    "armRegionName",
    "productName",
    "skuName",
    "meterName",
    "retailPrice",
    "unitOfMeasure",
    "currencyCode"
]].head(50)

,armRegionName,productName,skuName,meterName,retailPrice,unitOfMeasure,currencyCode
0,eastus,Virtual Machines NCdsxlRTX6kv6 Windows,NC320dsxlRTX6Kv6 Spot,NC320dsxlRTX6Kv6 Spot,6.261600,1 Hour,USD
1,eastus,DCasv5-series Linux,Standard_DC8as_v5 Low Priority,DC8as v5 Low Priority,0.068800,1 Hour,USD
2,eastus,Virtual Machines FXmdsv2 Series Windows,FX64mds v2 Low Priority,FX64mds v2 Low Priority,4.679000,1 Hour,USD
4,eastus,Virtual Machines HXrs Windows,HX176-48rs Low Priority,HX176-48rs Low Priority,6.694000,1 Hour,USD
7,eastus,Virtual Machines Ev5 Series,Standard_E8_v5 Low Priority,E8 v5 Low Priority,0.101000,1 Hour,USD
11,eastus,Virtual Machines GS Series Windows,GS2 Spot,GS2 Spot,0.225456,1 Hour,USD
14,eastus,Virtual Machines NC Promo Series Windows,NC24r,NC24r,2.846000,1 Hour,USD
15,eastus,Virtual Machines Msv3 Medium Memory Series Win...,Standard_M176s_v3 Spot,Standard_M176s_v3 Spot,4.565040,1 Hour,USD
16,eastus,Virtual Machines Mbsv4 series Linux,Standard_M32bs_v4,Standard_M32bs_v4,0.000000,1 Hour,USD
17,eastus,ECasv5-series Linux,Standard_EC96as_v5 Spot,EC96as_v5 Spot,1.145006,1 Hour,USD


In [5]:
def detect_gpu_type(text):
    text = str(text).upper()

    if "H200" in text:
        return "H200"
    if "H100" in text:
        return "H100"
    if "A100" in text:
        return "A100"
    if "L4" in text:
        return "L4"
    if "T4" in text:
        return "T4"
    if "V100" in text:
        return "V100"

    # Azure family-based inference
    if "ND" in text and "H100" not in text and "A100" not in text:
        return "Unknown ND-Series"
    if "NC" in text:
        return "Unknown NC-Series"
    if "NV" in text:
        return "Unknown NV-Series"

    return "Unknown"

azure_gpu["combined_text"] = (
    azure_gpu["productName"].astype(str) + " " +
    azure_gpu["skuName"].astype(str) + " " +
    azure_gpu["meterName"].astype(str)
)

azure_gpu["gpu_type"] = azure_gpu["combined_text"].apply(detect_gpu_type)

azure_gpu_v2 = azure_gpu[[
    "armRegionName",
    "productName",
    "skuName",
    "meterName",
    "retailPrice",
    "unitOfMeasure",
    "currencyCode",
    "gpu_type"
]].copy()

azure_gpu_v2 = azure_gpu_v2.rename(columns={
    "armRegionName": "region",
    "productName": "product_name",
    "skuName": "instance_type",
    "meterName": "meter_name",
    "retailPrice": "hourly_price_usd",
    "currencyCode": "currency"
})

azure_gpu_v2["provider"] = "Azure"
azure_gpu_v2["collection_date"] = COLLECTION_DATE

azure_gpu_v2 = azure_gpu_v2[
    azure_gpu_v2["hourly_price_usd"].notna()
]

azure_gpu_v2.to_csv(GPU_DIR / "azure_gpu_pricing_v2.csv", index=False)

azure_gpu_v2[[
    "provider",
    "region",
    "gpu_type",
    "instance_type",
    "hourly_price_usd",
    "unitOfMeasure"
]].head(50)

,provider,region,gpu_type,instance_type,hourly_price_usd,unitOfMeasure
0,Azure,eastus,Unknown ND-Series,NC320dsxlRTX6Kv6 Spot,6.261600,1 Hour
1,Azure,eastus,Unknown ND-Series,Standard_DC8as_v5 Low Priority,0.068800,1 Hour
2,Azure,eastus,Unknown ND-Series,FX64mds v2 Low Priority,4.679000,1 Hour
4,Azure,eastus,Unknown ND-Series,HX176-48rs Low Priority,6.694000,1 Hour
7,Azure,eastus,Unknown ND-Series,Standard_E8_v5 Low Priority,0.101000,1 Hour
11,Azure,eastus,Unknown ND-Series,GS2 Spot,0.225456,1 Hour
14,Azure,eastus,Unknown ND-Series,NC24r,2.846000,1 Hour
15,Azure,eastus,Unknown ND-Series,Standard_M176s_v3 Spot,4.565040,1 Hour
16,Azure,eastus,Unknown ND-Series,Standard_M32bs_v4,0.000000,1 Hour
17,Azure,eastus,Unknown ND-Series,Standard_EC96as_v5 Spot,1.145006,1 Hour


In [6]:
azure_gpu_v2.groupby("gpu_type").agg(
    records=("instance_type", "count"),
    unique_instances=("instance_type", "nunique"),
    min_price=("hourly_price_usd", "min"),
    avg_price=("hourly_price_usd", "mean"),
    max_price=("hourly_price_usd", "max")
).reset_index().sort_values("avg_price", ascending=False)

,gpu_type,records,unique_instances,min_price,avg_price,max_price
1,H100,36,18,1.396000,36.412879,102.736
0,A100,44,23,0.735000,12.628806,37.186
5,Unknown ND-Series,5266,4061,0.000000,5.312124,451.616
4,Unknown NC-Series,74,74,0.008940,3.713739,16.588
2,L4,42,21,0.063386,1.834053,6.384
3,T4,24,12,0.105000,1.305712,7.296
6,Unknown NV-Series,41,41,0.043058,1.017873,7.176


In [7]:
azure_gpu_v2[
    azure_gpu_v2["gpu_type"]=="Unknown ND-Series"
][[
    "product_name",
    "instance_type",
    "meter_name",
    "hourly_price_usd"
]].head(50)

,product_name,instance_type,meter_name,hourly_price_usd
0,Virtual Machines NCdsxlRTX6kv6 Windows,NC320dsxlRTX6Kv6 Spot,NC320dsxlRTX6Kv6 Spot,6.261600
1,DCasv5-series Linux,Standard_DC8as_v5 Low Priority,DC8as v5 Low Priority,0.068800
2,Virtual Machines FXmdsv2 Series Windows,FX64mds v2 Low Priority,FX64mds v2 Low Priority,4.679000
4,Virtual Machines HXrs Windows,HX176-48rs Low Priority,HX176-48rs Low Priority,6.694000
7,Virtual Machines Ev5 Series,Standard_E8_v5 Low Priority,E8 v5 Low Priority,0.101000
11,Virtual Machines GS Series Windows,GS2 Spot,GS2 Spot,0.225456
14,Virtual Machines NC Promo Series Windows,NC24r,NC24r,2.846000
15,Virtual Machines Msv3 Medium Memory Series Win...,Standard_M176s_v3 Spot,Standard_M176s_v3 Spot,4.565040
16,Virtual Machines Mbsv4 series Linux,Standard_M32bs_v4,Standard_M32bs_v4,0.000000
17,ECasv5-series Linux,Standard_EC96as_v5 Spot,EC96as_v5 Spot,1.145006


In [8]:
azure_gpu_v2[
    azure_gpu_v2["gpu_type"]=="Unknown ND-Series"
]["instance_type"].value_counts().head(30)

instance_type
Standard_D2as_v5                   3
Standard_D64as_v5 Low Priority     3
Standard_D64as_v5                  3
Standard_D8as_v5                   3
Standard_D32ads_v5                 3
Standard_D4as_v5 Low Priority      3
Standard_D4ads_v5 Low Priority     3
Standard_D8ads_v5                  3
Standard_D2as_v5 Low Priority      3
Standard_D64ads_v5                 3
Standard_D96ads_v5 Low Priority    3
Standard_D16ads_v5 Low Priority    3
Standard_D48as_v5                  3
Standard_D96as_v5                  3
Standard_D2ads_v5 Low Priority     3
Standard_D2ads_v5                  3
Standard_D32as_v5 Low Priority     3
Standard_D48ads_v5 Low Priority    3
Standard_D64ads_v5 Low Priority    3
Standard_D8as_v5 Low Priority      3
Standard_D4ads_v5                  3
Standard_D96as_v5 Low Priority     3
Standard_D32ads_v5 Low Priority    3
Standard_D96ads_v5                 3
Standard_D16as_v5                  3
Standard_D16ads_v5                 3
Standard_D48as_v5 Low Pr

In [9]:
azure_gpu_v2["instance_type"].str.contains(
    "A100|H100|H200|A10|L4|T4|V100",
    case=False,
    na=False
).sum()

np.int64(185)

In [10]:
len(azure_gpu_v2)

5527

In [11]:
azure_gpu_keywords = [
    "A100",
    "H100",
    "H200",
    "A10",
    "L4",
    "T4",
    "V100",
    "MI300"
]

In [12]:
pattern = "|".join(azure_gpu_keywords)

azure_gpu_clean = azure_raw[
    azure_raw["productName"].str.contains(pattern, case=False, na=False)
    |
    azure_raw["skuName"].str.contains(pattern, case=False, na=False)
    |
    azure_raw["meterName"].str.contains(pattern, case=False, na=False)
].copy()

In [13]:
azure_gpu_clean[
    [
        "productName",
        "skuName",
        "meterName",
        "retailPrice"
    ]
].head(50)

,productName,skuName,meterName,retailPrice
42,Virtual Machines Lsv3 Series,Standard_L48s_v3,L48s v3,4.176000
49,Virtual Machines Lasv4 Series,L4as v4 Spot,L4as v4 Spot,0.072102
59,NCads A100 v4 Series Linux,Standard_NC48ads_A100_v4 Low Priority,NC48ads_A100_v4 Low Priority,1.469000
67,Virtual Machines LSv2 Series,L48s v2,L48s v2,3.744000
76,Virtual Machines NVadsA10v5 Series Windows,Standard_NV36ads_A10_v5,NV36ads A10 v5,4.856000
79,Virtual Machines NVadsA10v5 Series Windows,Standard_NV36adms_A10_v5,NV36adms A10 v5,6.176000
101,Virtual Machines LSv2 Series Windows,L48s v2,L48s v2,5.952000
117,Virtual Machines NDsr H100 v5 Series,ND96isrH100v5,ND96isrH100v5,98.320000
144,Virtual Machines NVadsA10v5 Series,Standard_NV6ads_A10_v5,NV6ads A10 v5,0.454000
151,Virtual Machines Lsv4 Series,L48s v4 Spot,L48s v4 Spot,0.876542


In [14]:
azure_gpu_clean.shape

(188, 21)

In [15]:
azure_gpu_clean[[
    "productName",
    "skuName",
    "retailPrice"
]].head(30)

,productName,skuName,retailPrice
42,Virtual Machines Lsv3 Series,Standard_L48s_v3,4.176000
49,Virtual Machines Lasv4 Series,L4as v4 Spot,0.072102
59,NCads A100 v4 Series Linux,Standard_NC48ads_A100_v4 Low Priority,1.469000
67,Virtual Machines LSv2 Series,L48s v2,3.744000
76,Virtual Machines NVadsA10v5 Series Windows,Standard_NV36ads_A10_v5,4.856000
79,Virtual Machines NVadsA10v5 Series Windows,Standard_NV36adms_A10_v5,6.176000
101,Virtual Machines LSv2 Series Windows,L48s v2,5.952000
117,Virtual Machines NDsr H100 v5 Series,ND96isrH100v5,98.320000
144,Virtual Machines NVadsA10v5 Series,Standard_NV6ads_A10_v5,0.454000
151,Virtual Machines Lsv4 Series,L48s v4 Spot,0.876542


In [16]:
gpu_keywords = [
    "A100",
    "H100",
    "H200",
    "A10",
    "T4",
    "V100",
    "MI300"
]

In [17]:
def gpu_type(text):
    text = str(text).upper()

    if "H200" in text:
        return "H200"
    elif "H100" in text:
        return "H100"
    elif "A100" in text:
        return "A100"
    elif "A10" in text:
        return "A10"
    elif "T4" in text:
        return "T4"
    elif "V100" in text:
        return "V100"
    elif "MI300" in text:
        return "MI300"
    else:
        return "Other"

In [19]:
azure_gpu_clean.shape

(188, 21)

In [20]:
azure_gpu_clean.head()

,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,...,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,effectiveEndDate
42,USD,0.0,4.176000,4.176000,eastus,US East,2022-06-01T00:00:00Z,01560ec2-7e2d-5579-841e-ef06c3bd2d06,L48s v3,DZH318Z08NR7,...,Virtual Machines Lsv3 Series,Standard_L48s_v3,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_L48s_v3,NaN
49,USD,0.0,0.072102,0.072102,eastus,US East,2026-06-01T00:00:00Z,01a1f277-14d6-5ea7-a3ed-de12bccf1189,L4as v4 Spot,DZH318Z0M2SF,...,Virtual Machines Lasv4 Series,L4as v4 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_L4as_v4,NaN
59,USD,0.0,1.469000,1.469000,eastus,US East,2022-06-01T00:00:00Z,01f6dc96-b544-5fd4-9e2f-18db001a33d5,NC48ads_A100_v4 Low Priority,DZH318Z09TGJ,...,NCads A100 v4 Series Linux,Standard_NC48ads_A100_v4 Low Priority,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_NC48ads_A100_v4,NaN
67,USD,0.0,3.744000,3.744000,eastus,US East,2025-10-01T00:00:00Z,025402b3-c7f6-413f-99b8-cc2ffb074c3e,L48s v2,DZH318Z0BQM3,...,Virtual Machines LSv2 Series,L48s v2,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_L48s_v2,NaN
76,USD,0.0,4.856000,4.856000,eastus,US East,2022-04-01T00:00:00Z,02a36816-a172-5706-a433-54871e69f893,NV36ads A10 v5,DZH318Z09JCM,...,Virtual Machines NVadsA10v5 Series Windows,Standard_NV36ads_A10_v5,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_NV36ads_A10_v5,NaN


In [21]:
dir()

['COLLECTION_DATE',
 'GPU_DIR',
 'In',
 'Out',
 'Path',
 'TARGET_GPUS',
 'TARGET_REGIONS',
 '_',
 '_10',
 '_13',
 '_14',
 '_15',
 '_19',
 '_20',
 '_3',
 '_4',
 '_5',
 '_6',
 '_7',
 '_8',
 '_9',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__session__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i10',
 '_i11',
 '_i12',
 '_i13',
 '_i14',
 '_i15',
 '_i16',
 '_i17',
 '_i18',
 '_i19',
 '_i2',
 '_i20',
 '_i21',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_i9',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'azure_filter',
 'azure_gpu',
 'azure_gpu_clean',
 'azure_gpu_keywords',
 'azure_gpu_v2',
 'azure_raw',
 'date',
 'detect_gpu_type',
 'exit',
 'fetch_azure_retail_prices',
 'get_ipython',
 'gpu_keywords',
 'gpu_type',
 'json',
 'open',
 'pattern',
 'pd',
 'quit',
 're',
 'requests']

In [22]:
# =========================
# Build Azure GPU v2 Final
# =========================

azure_gpu_clean["combined_text"] = (
    azure_gpu_clean["productName"].astype(str) + " " +
    azure_gpu_clean["skuName"].astype(str) + " " +
    azure_gpu_clean["meterName"].astype(str)
)

def detect_gpu_type_v2(text):
    text = str(text).upper()

    if "H200" in text:
        return "H200"
    elif "H100" in text:
        return "H100"
    elif "A100" in text:
        return "A100"
    elif "MI300" in text:
        return "MI300"
    elif "A10" in text:
        return "A10"
    elif "T4" in text:
        return "T4"
    elif "V100" in text:
        return "V100"
    else:
        return "Other"

azure_gpu_clean["gpu_type"] = azure_gpu_clean["combined_text"].apply(detect_gpu_type_v2)

# 移除 Azure L-series storage VM，不是 NVIDIA L4 GPU
bad_patterns = [
    "LSV2",
    "LSV3",
    "LSV4",
    "LASV",
    "L48",
    "L64",
    "L80",
    "L96",
    "L4S"
]

bad_pattern = "|".join(bad_patterns)

azure_gpu_clean = azure_gpu_clean[
    ~azure_gpu_clean["combined_text"].str.contains(
        bad_pattern,
        case=False,
        na=False
    )
].copy()

# 移除 Spot / Low Priority，只保留標準 on-demand 價格
azure_gpu_clean = azure_gpu_clean[
    ~azure_gpu_clean["combined_text"].str.contains(
        "Spot|Low Priority",
        case=False,
        na=False
    )
].copy()

# 建立 final table
azure_gpu_v2_final = azure_gpu_clean.rename(columns={
    "armRegionName": "region",
    "productName": "product_name",
    "skuName": "instance_type",
    "meterName": "meter_name",
    "retailPrice": "hourly_price_usd",
    "currencyCode": "currency"
})[
    [
        "region",
        "product_name",
        "instance_type",
        "meter_name",
        "gpu_type",
        "hourly_price_usd",
        "currency",
        "unitOfMeasure"
    ]
].copy()

azure_gpu_v2_final["provider"] = "Azure"
azure_gpu_v2_final["collection_date"] = COLLECTION_DATE

azure_gpu_v2_final = azure_gpu_v2_final[
    azure_gpu_v2_final["hourly_price_usd"].notna() &
    (azure_gpu_v2_final["hourly_price_usd"] > 0) &
    (azure_gpu_v2_final["gpu_type"] != "Other")
].copy()

azure_gpu_v2_final = azure_gpu_v2_final.sort_values(
    ["gpu_type", "hourly_price_usd"]
).reset_index(drop=True)

print(azure_gpu_v2_final.shape)
azure_gpu_v2_final.head(50)

(52, 10)


,region,product_name,instance_type,meter_name,gpu_type,hourly_price_usd,currency,unitOfMeasure,provider,collection_date
0,eastus,Virtual Machines NVadsA10v5 Series,Standard_NV6ads_A10_v5,NV6ads A10 v5,A10,0.454,USD,1 Hour,Azure,2026-06-05
1,eastus,Virtual Machines NVadsA10v5 Series Windows,Standard_NV6ads_A10_v5,NV6ads A10 v5,A10,0.730,USD,1 Hour,Azure,2026-06-05
2,eastus,Virtual Machines A Series,A10,A10,A10,0.780,USD,1 Hour,Azure,2026-06-05
3,eastus,Virtual Machines NVadsA10v5 Series,Standard_NV12ads_A10_v5,NV12ads A10 v5,A10,0.908,USD,1 Hour,Azure,2026-06-05
4,eastus,Virtual Machines A Series Windows,A10,A10,A10,1.173,USD,1 Hour,Azure,2026-06-05
5,eastus,Virtual Machines NVadsA10v5 Series Windows,Standard_NV12ads_A10_v5,NV12ads A10 v5,A10,1.460,USD,1 Hour,Azure,2026-06-05
6,eastus,Virtual Machines NVadsA10v5 Series,Standard_NV18ads_A10_v5,NV18ads A10 v5,A10,1.600,USD,1 Hour,Azure,2026-06-05
7,eastus,Virtual Machines NVadsA10v5 Series Windows,Standard_NV18ads_A10_v5,NV18ads A10 v5,A10,2.428,USD,1 Hour,Azure,2026-06-05
8,eastus,Virtual Machines NVadsA10v5 Series,Standard_NV36ads_A10_v5,NV36ads A10 v5,A10,3.200,USD,1 Hour,Azure,2026-06-05
9,eastus,Virtual Machines NVadsA10v5 Series,Standard_NV36adms_A10_v5,NV36adms A10 v5,A10,4.520,USD,1 Hour,Azure,2026-06-05


In [23]:
azure_gpu_summary_check = (
    azure_gpu_v2_final
    .groupby("gpu_type")
    .agg(
        records=("instance_type", "count"),
        unique_instances=("instance_type", "nunique"),
        min_price=("hourly_price_usd", "min"),
        avg_price=("hourly_price_usd", "mean"),
        max_price=("hourly_price_usd", "max")
    )
    .reset_index()
    .sort_values("avg_price", ascending=False)
)

azure_gpu_summary_check

,gpu_type,records,unique_instances,min_price,avg_price,max_price
2,H100,14,7,6.980,70.977714,102.736
1,A100,16,9,3.673,23.915375,37.186
0,A10,14,7,0.454,3.188357,9.832
3,T4,8,4,0.526,2.237500,7.296


In [25]:
from pathlib import Path

OUTPUT_DIR = Path("gpu_dataset_v2")
OUTPUT_DIR.mkdir(exist_ok=True)

azure_gpu_v2_final.to_csv(
    OUTPUT_DIR / "azure_gpu_pricing_v2_clean.csv",
    index=False
)

azure_gpu_summary_check.to_csv(
    OUTPUT_DIR / "azure_gpu_summary_v2_clean.csv",
    index=False
)

print("Done!")

Done!


In [26]:
import os

print(os.getcwd())

C:\Users\user\Cloud Project


In [27]:
import pandas as pd
import requests
import json
from pathlib import Path
from datetime import date

GPU_DIR = Path("gpu_dataset_v2")
GPU_DIR.mkdir(exist_ok=True)

COLLECTION_DATE = date.today().isoformat()

print("Ready")

Ready


In [28]:
url = "https://pricing.us-east-1.amazonaws.com/offers/v1.0/aws/AmazonEC2/current/index.json"

print("Downloading...")

r = requests.get(url, timeout=300)

print(r.status_code)

data = r.json()

print(data.keys())

Downloading...
200
dict_keys(['formatVersion', 'disclaimer', 'offerCode', 'version', 'publicationDate', 'products', 'terms', 'attributesList'])


In [29]:
products = pd.DataFrame(
    data["products"]
).T.reset_index(drop=True)

products.shape

(2006559, 3)

In [30]:
products.columns

Index(['sku', 'productFamily', 'attributes'], dtype='object')

In [31]:
products["attributes"].iloc[0]

{'servicecode': 'AmazonEC2',
 'location': 'Asia Pacific (Malaysia)',
 'locationType': 'AWS Region',
 'instanceType': 'c7gd.medium',
 'currentGeneration': 'Yes',
 'instanceFamily': 'Compute optimized',
 'vcpu': '1',
 'physicalProcessor': 'AWS Graviton3 Processor',
 'clockSpeed': '2.5 GHz',
 'memory': '2 GiB',
 'storage': '1 x 59 NVMe SSD',
 'networkPerformance': 'Up to 12500 Megabit',
 'processorArchitecture': '64-bit',
 'tenancy': 'Shared',
 'operatingSystem': 'RHEL',
 'licenseModel': 'No License required',
 'usagetype': 'APS7-Reservation:c7gd.medium',
 'operation': 'RunInstances:0010',
 'availabilityzone': 'NA',
 'capacitystatus': 'AllocatedCapacityReservation',
 'classicnetworkingsupport': 'false',
 'dedicatedEbsThroughput': 'Up to 10000 Mbps',
 'dedicatedEbsThroughputDescription': '315 Mbps',
 'ecu': 'NA',
 'enhancedNetworkingSupported': 'Yes',
 'gpuMemory': 'NA',
 'instanceFamilyCategory': 'Compute Optimized',
 'instancesku': '7CQ8E9D98XGDXC9R',
 'intelAvxAvailable': 'No',
 'intelA

In [32]:
attrs = pd.json_normalize(products["attributes"])

aws_products = pd.concat(
    [
        products[["sku", "productFamily"]],
        attrs
    ],
    axis=1
)

print(aws_products.shape)
aws_products.head()

(2006559, 79)


,sku,productFamily,servicecode,location,locationType,instanceType,currentGeneration,instanceFamily,vcpu,physicalProcessor,...,volumeType,maxVolumeSize,maxIopsvolume,maxIopsBurstPerformance,maxThroughputvolume,instance,snapshotarchivefeetype,ebsOptimized,elasticGraphicsType,instanceCapacity-10xlarge
0,RSH2Y67N4H4CFBQ4,Compute Instance,AmazonEC2,Asia Pacific (Malaysia),AWS Region,c7gd.medium,Yes,Compute optimized,1,AWS Graviton3 Processor,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,YHBHTMRVRUX8M33Y,Compute Instance,AmazonEC2,Asia Pacific (Malaysia),AWS Region,m7i.large,Yes,General purpose,2,Intel Xeon Scalable (Sapphire Rapids),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JSEGQBVEG2Y4NYJR,Compute Instance,AmazonEC2,Asia Pacific (Thailand),AWS Region,c6in.24xlarge,Yes,Compute optimized,96,Intel Xeon 8375C (Ice Lake),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,646B9QM23GWYVTAK,Compute Instance,AmazonEC2,Africa (Cape Town),AWS Region,c7i.xlarge,Yes,Compute optimized,4,Intel Xeon Scalable (Sapphire Rapids),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SSMSW7PJQYNKBTE4,Compute Instance,AmazonEC2,AWS GovCloud (US-West),AWS Region,m6i.large,Yes,General purpose,2,Intel Xeon 8375C (Ice Lake),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
aws_ec2_us_east = aws_products[
    (aws_products["regionCode"] == "us-east-1") &
    (aws_products["operatingSystem"] == "Linux") &
    (aws_products["tenancy"] == "Shared") &
    (aws_products["marketoption"] == "OnDemand") &
    (aws_products["capacitystatus"] == "Used")
].copy()

print(aws_ec2_us_east.shape)

aws_ec2_us_east[[
    "sku",
    "instanceType",
    "instanceFamily",
    "vcpu",
    "memory",
    "gpuMemory",
    "physicalProcessor",
    "operation"
]].head(20)

(3817, 79)


,sku,instanceType,instanceFamily,vcpu,memory,gpuMemory,physicalProcessor,operation
377,5XNK75YHGW5FJNRH,g6.24xlarge,GPU instance,96,384 GiB,96 GB,AMD EPYC 7R13 Processor,RunInstances
1098,5G4TA8Z4MUKE6MJB,m5.xlarge,General purpose,4,16 GiB,NA,Intel Xeon Platinum 8175,RunInstances
1340,QQK6UQEYHSB9KXCM,r5n.8xlarge,Memory optimized,32,256 GiB,NA,Intel Xeon Platinum 8259 (Cascade Lake),RunInstances:0100
4365,283SQKP85WVMA6PM,m8azn.6xlarge,General purpose,24,96 GiB,NA,AMD EPYC 9575F,RunInstances:0200
5005,HAEHM4SF35ZHXY39,m8ib.48xlarge,General purpose,192,768.0 GiB,NA,Intel Xeon Scalable (Granite Rapids),RunInstances:0004
5068,9A947RRUWRTYKQXH,r6in.24xlarge,Memory optimized,96,768 GiB,NA,Intel Xeon 8375C (Ice Lake),RunInstances:0100
5115,23B83J99HS4RPGHB,c7i.xlarge,Compute optimized,4,8 GiB,NA,Intel Xeon Scalable (Sapphire Rapids),RunInstances:0100
5310,HEM4NPMZ3U4FEZ6G,c8gb.xlarge,Compute optimized,4,8 GiB,NA,AWS Graviton4 Processor,RunInstances
6245,C6RRFRMEVTTW77Y4,c8ine.4xlarge,Compute optimized,16,32 GiB,NA,Intel Xeon Scalable (Granite Rapids),RunInstances:0004
6348,X6BBF3V4Z2M2HNM3,m8a.medium,General purpose,1,4 GiB,NA,AMD EPYC 9R45 Processor,RunInstances:0200


In [34]:
gpu_family_keywords = [
    "p2", "p3", "p4", "p5", "p6",
    "g2", "g3", "g4", "g5", "g6", "g7",
    "inf1", "inf2",
    "trn1", "trn2",
    "dl1"
]

gpu_pattern = "|".join([f"^{x}\\." for x in gpu_family_keywords])

aws_gpu_candidates = aws_ec2_us_east[
    aws_ec2_us_east["instanceType"].str.contains(
        gpu_pattern,
        case=False,
        na=False,
        regex=True
    )
].copy()

print(aws_gpu_candidates.shape)

aws_gpu_candidates[[
    "sku",
    "instanceType",
    "instanceFamily",
    "vcpu",
    "memory",
    "gpuMemory",
    "physicalProcessor",
    "operation"
]].head(50)

(79, 79)


,sku,instanceType,instanceFamily,vcpu,memory,gpuMemory,physicalProcessor,operation
377,5XNK75YHGW5FJNRH,g6.24xlarge,GPU instance,96,384 GiB,96 GB,AMD EPYC 7R13 Processor,RunInstances
41234,GRY9GJ9Y7T28MXVR,g6.4xlarge,GPU instance,16,64 GiB,24 GB,AMD EPYC 7R13 Processor,RunInstances:0004
41314,ZYF8F3CMDWWA922R,g5.8xlarge,GPU instance,32,128 GiB,24 GB,AMD EPYC 7R32,RunInstances
42228,FSFQ4X3WCNGZ7RKR,inf1.6xlarge,Machine Learning ASIC Instances,24,48 GiB,NA,Intel Xeon Platinum 8275CL (Cascade Lake),RunInstances:0004
81471,S8JTWNNXNMNADCBB,inf1.24xlarge,Machine Learning ASIC Instances,96,192 GiB,NA,Intel Xeon Platinum 8275CL (Cascade Lake),RunInstances
119743,4GQWNPC9K2PZAY97,g5.4xlarge,GPU instance,16,64 GiB,24 GB,AMD EPYC 7R32,RunInstances
138892,PDSYKDKX8RBFG4UQ,g6.48xlarge,GPU instance,192,768 GiB,192 GB,AMD EPYC 7R13 Processor,RunInstances:0004
208815,FX9TUYTJKEENVBSR,g5.2xlarge,GPU instance,8,32 GiB,24 GB,AMD EPYC 7R32,RunInstances
226332,SJ6GAZ9TTCDWW8PG,g2.8xlarge,GPU instance,32,60 GiB,NA,Intel Xeon E5-2670 (Sandy Bridge),RunInstances:0100
228462,K9VKFEBARNNMVRXV,p5.4xlarge,GPU instance,16,256 GiB,80 GB HBM3,AMD EPYC 7R13 Processor,RunInstances


In [35]:
aws_gpu_candidates["instanceType"].str.extract(r"^([a-z0-9]+)")[0].value_counts()

0
g6      32
inf1    16
g5       8
g2       5
inf2     4
p2       3
g3       3
p3       3
p5       2
trn1     2
dl1      1
Name: count, dtype: int64

In [36]:
# =========================
# AWS On-Demand Price Parser
# =========================

ondemand_terms = data["terms"]["OnDemand"]

price_rows = []

for sku, term_data in ondemand_terms.items():
    if sku not in set(aws_gpu_candidates["sku"]):
        continue

    for term_code, term in term_data.items():
        price_dimensions = term.get("priceDimensions", {})

        for price_dim_code, price_dim in price_dimensions.items():
            price_per_unit = price_dim.get("pricePerUnit", {})
            usd_price = price_per_unit.get("USD")

            if usd_price is None:
                continue

            price_rows.append({
                "sku": sku,
                "term_code": term_code,
                "price_dim_code": price_dim_code,
                "unit": price_dim.get("unit"),
                "description": price_dim.get("description"),
                "hourly_price_usd": float(usd_price)
            })

aws_price_df = pd.DataFrame(price_rows)

print(aws_price_df.shape)
aws_price_df.head(20)

(79, 6)


,sku,term_code,price_dim_code,unit,description,hourly_price_usd
0,TP4UD6FKK6A8GEDA,TP4UD6FKK6A8GEDA.JRTCKXETXF,TP4UD6FKK6A8GEDA.JRTCKXETXF.6YS6EN2CT7,Hrs,$13.10904 per On Demand Linux dl1.24xlarge Ins...,13.10904
1,48VURD6MVAZ3M5JX,48VURD6MVAZ3M5JX.JRTCKXETXF,48VURD6MVAZ3M5JX.JRTCKXETXF.6YS6EN2CT7,Hrs,$0.650 per On Demand Linux g2.2xlarge Instance...,0.65000
2,ND5FJH5H862EPPXB,ND5FJH5H862EPPXB.JRTCKXETXF,ND5FJH5H862EPPXB.JRTCKXETXF.6YS6EN2CT7,Hrs,$1.61 per On Demand Linux with SQL Std g2.2xla...,1.61000
3,K4XA4PJA2UTCNQKX,K4XA4PJA2UTCNQKX.JRTCKXETXF,K4XA4PJA2UTCNQKX.JRTCKXETXF.6YS6EN2CT7,Hrs,$0.785 per On Demand Linux with SQL Web g2.2xl...,0.78500
4,A67CJDV9B3YBP6N6,A67CJDV9B3YBP6N6.JRTCKXETXF,A67CJDV9B3YBP6N6.JRTCKXETXF.6YS6EN2CT7,Hrs,$2.6 per On Demand Linux g2.8xlarge Instance Hour,2.60000
5,SJ6GAZ9TTCDWW8PG,SJ6GAZ9TTCDWW8PG.JRTCKXETXF,SJ6GAZ9TTCDWW8PG.JRTCKXETXF.6YS6EN2CT7,Hrs,$14.60 per On Demand Linux with SQL Server Ent...,14.60000
6,3H63S5QV423QAHHQ,3H63S5QV423QAHHQ.JRTCKXETXF,3H63S5QV423QAHHQ.JRTCKXETXF.6YS6EN2CT7,Hrs,$4.56 per On Demand Linux g3.16xlarge Instance...,4.56000
7,SQ37ZQ2CZ2H95VDC,SQ37ZQ2CZ2H95VDC.JRTCKXETXF,SQ37ZQ2CZ2H95VDC.JRTCKXETXF.6YS6EN2CT7,Hrs,$1.14 per On Demand Linux g3.4xlarge Instance ...,1.14000
8,RSMWKBGGTAAEV4RH,RSMWKBGGTAAEV4RH.JRTCKXETXF,RSMWKBGGTAAEV4RH.JRTCKXETXF.6YS6EN2CT7,Hrs,$2.28 per On Demand Linux g3.8xlarge Instance ...,2.28000
9,UH59G55UJ94HZTJJ,UH59G55UJ94HZTJJ.JRTCKXETXF,UH59G55UJ94HZTJJ.JRTCKXETXF.6YS6EN2CT7,Hrs,$5.672 per On Demand Linux g5.12xlarge Instanc...,5.67200


In [37]:
aws_gpu_pricing = aws_gpu_candidates.merge(
    aws_price_df,
    on="sku",
    how="left"
)

aws_gpu_pricing[[
    "sku",
    "instanceType",
    "instanceFamily",
    "vcpu",
    "memory",
    "gpuMemory",
    "physicalProcessor",
    "hourly_price_usd",
    "unit",
    "description"
]].head(50)

,sku,instanceType,instanceFamily,vcpu,memory,gpuMemory,physicalProcessor,hourly_price_usd,unit,description
0,5XNK75YHGW5FJNRH,g6.24xlarge,GPU instance,96,384 GiB,96 GB,AMD EPYC 7R13 Processor,6.67520,Hrs,$6.6752 per On Demand Linux g6.24xlarge Instan...
1,GRY9GJ9Y7T28MXVR,g6.4xlarge,GPU instance,16,64 GiB,24 GB,AMD EPYC 7R13 Processor,3.24320,Hrs,$3.2432 per On Demand Linux with SQL Std g6.4x...
2,ZYF8F3CMDWWA922R,g5.8xlarge,GPU instance,32,128 GiB,24 GB,AMD EPYC 7R32,2.44800,Hrs,$2.448 per On Demand Linux g5.8xlarge Instance...
3,FSFQ4X3WCNGZ7RKR,inf1.6xlarge,Machine Learning ASIC Instances,24,48 GiB,NA,Intel Xeon Platinum 8275CL (Cascade Lake),4.06000,Hrs,$4.06 per On Demand Linux with SQL Std inf1.6x...
4,S8JTWNNXNMNADCBB,inf1.24xlarge,Machine Learning ASIC Instances,96,192 GiB,NA,Intel Xeon Platinum 8275CL (Cascade Lake),4.72100,Hrs,$4.721 per On Demand Linux inf1.24xlarge Insta...
5,4GQWNPC9K2PZAY97,g5.4xlarge,GPU instance,16,64 GiB,24 GB,AMD EPYC 7R32,1.62400,Hrs,$1.624 per On Demand Linux g5.4xlarge Instance...
6,PDSYKDKX8RBFG4UQ,g6.48xlarge,GPU instance,192,768 GiB,192 GB,AMD EPYC 7R13 Processor,36.39040,Hrs,$36.3904 per On Demand Linux with SQL Std g6.4...
7,FX9TUYTJKEENVBSR,g5.2xlarge,GPU instance,8,32 GiB,24 GB,AMD EPYC 7R32,1.21200,Hrs,$1.212 per On Demand Linux g5.2xlarge Instance...
8,SJ6GAZ9TTCDWW8PG,g2.8xlarge,GPU instance,32,60 GiB,NA,Intel Xeon E5-2670 (Sandy Bridge),14.60000,Hrs,$14.60 per On Demand Linux with SQL Server Ent...
9,K9VKFEBARNNMVRXV,p5.4xlarge,GPU instance,16,256 GiB,80 GB HBM3,AMD EPYC 7R13 Processor,6.88000,Hrs,$6.88 per On Demand Linux p5.4xlarge Instance ...


In [38]:
aws_gpu_pricing["hourly_price_usd"].isna().sum(), len(aws_gpu_pricing)

(np.int64(0), 79)

In [39]:
aws_gpu_pricing.groupby(
    aws_gpu_pricing["instanceType"].str.extract(r"^([a-z0-9]+)")[0]
).agg(
    records=("instanceType", "count"),
    unique_instances=("instanceType", "nunique"),
    min_price=("hourly_price_usd", "min"),
    avg_price=("hourly_price_usd", "mean"),
    max_price=("hourly_price_usd", "max")
).reset_index().sort_values("avg_price", ascending=False)

,0,records,unique_instances,min_price,avg_price,max_price
9,p5,2,2,6.88000,30.960000,55.04000
8,p3,3,3,3.06000,13.260000,24.48000
0,dl1,1,1,13.10904,13.109040,13.10904
4,g6,32,8,0.80480,11.501563,85.35040
10,trn1,2,2,1.34375,11.421875,21.50000
7,p2,3,3,0.90000,7.500000,14.40000
5,inf1,16,4,0.22800,5.846000,40.72100
6,inf2,4,4,0.75820,5.549490,12.98127
3,g5,8,8,1.00600,5.061250,16.28800
1,g2,5,2,0.65000,4.049000,14.60000


In [40]:
aws_gpu_map = {
    "g6":"L4",
    "g5":"A10G",
    "p5":"H100",
    "p4":"A100",
    "p3":"V100",
    "p2":"K80",
    "g4dn":"T4",
    "g3":"M60",
    "inf1":"Inferentia",
    "inf2":"Inferentia2",
    "trn1":"Trainium",
    "trn2":"Trainium2",
    "dl1":"Gaudi"
}

In [41]:
aws_gpu_pricing["gpu_type"] = (
    aws_gpu_pricing["instanceType"]
    .str.extract(r"^([a-z0-9]+)")[0]
    .map(aws_gpu_map)
)

In [42]:
aws_gpu_final = aws_gpu_pricing[
    aws_gpu_pricing["gpu_type"].notna()
].copy()

In [43]:
aws_gpu_final.groupby("gpu_type").agg(
    records=("instanceType","count"),
    unique_instances=("instanceType","nunique"),
    min_price=("hourly_price_usd","min"),
    avg_price=("hourly_price_usd","mean"),
    max_price=("hourly_price_usd","max")
).reset_index().sort_values(
    "avg_price",
    ascending=False
)

,gpu_type,records,unique_instances,min_price,avg_price,max_price
2,H100,2,2,6.88000,30.960000,55.04000
9,V100,3,3,3.06000,13.260000,24.48000
1,Gaudi,1,1,13.10904,13.109040,13.10904
6,L4,32,8,0.80480,11.501563,85.35040
8,Trainium,2,2,1.34375,11.421875,21.50000
5,K80,3,3,0.90000,7.500000,14.40000
3,Inferentia,16,4,0.22800,5.846000,40.72100
4,Inferentia2,4,4,0.75820,5.549490,12.98127
0,A10G,8,8,1.00600,5.061250,16.28800
7,M60,3,3,1.14000,2.660000,4.56000


In [44]:
dir()

['COLLECTION_DATE',
 'GPU_DIR',
 'In',
 'OUTPUT_DIR',
 'Out',
 'Path',
 'TARGET_GPUS',
 'TARGET_REGIONS',
 '_',
 '_10',
 '_13',
 '_14',
 '_15',
 '_19',
 '_20',
 '_21',
 '_22',
 '_23',
 '_29',
 '_3',
 '_30',
 '_31',
 '_32',
 '_33',
 '_34',
 '_35',
 '_36',
 '_37',
 '_38',
 '_39',
 '_4',
 '_43',
 '_5',
 '_6',
 '_7',
 '_8',
 '_9',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__session__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i10',
 '_i11',
 '_i12',
 '_i13',
 '_i14',
 '_i15',
 '_i16',
 '_i17',
 '_i18',
 '_i19',
 '_i2',
 '_i20',
 '_i21',
 '_i22',
 '_i23',
 '_i24',
 '_i25',
 '_i26',
 '_i27',
 '_i28',
 '_i29',
 '_i3',
 '_i30',
 '_i31',
 '_i32',
 '_i33',
 '_i34',
 '_i35',
 '_i36',
 '_i37',
 '_i38',
 '_i39',
 '_i4',
 '_i40',
 '_i41',
 '_i42',
 '_i43',
 '_i44',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_i9',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'attrs',
 'aws_ec2_us_east',
 'aws_gpu_candidates',
 'aws_gpu_final',
 'aws_gpu_map',
 'aws_gpu_pricin

In [45]:
from pathlib import Path

GPU_DIR = Path(r"C:\Users\user\Cloud Project\gpu_dataset_v2")
GPU_DIR.mkdir(parents=True, exist_ok=True)

In [48]:
[x for x in dir() if "aws" in x.lower()]

['aws_ec2_us_east',
 'aws_gpu_candidates',
 'aws_gpu_final',
 'aws_gpu_map',
 'aws_gpu_pricing',
 'aws_price_df',
 'aws_products']

In [49]:
from pathlib import Path

GPU_DIR = Path(r"C:\Users\user\Cloud Project\gpu_dataset_v2")
GPU_DIR.mkdir(parents=True, exist_ok=True)

aws_gpu_pricing_v2_clean = aws_gpu_final.rename(columns={
    "instanceType": "instance_type",
    "regionCode": "region",
    "vcpu": "vcpu",
    "memory": "memory",
    "gpuMemory": "gpu_memory",
    "physicalProcessor": "physical_processor"
}).copy()

aws_gpu_pricing_v2_clean["provider"] = "AWS"
aws_gpu_pricing_v2_clean["collection_date"] = COLLECTION_DATE
aws_gpu_pricing_v2_clean["currency"] = "USD"

keep_cols = [
    "provider",
    "region",
    "gpu_type",
    "instance_type",
    "vcpu",
    "memory",
    "gpu_memory",
    "physical_processor",
    "hourly_price_usd",
    "currency",
    "collection_date"
]

aws_gpu_pricing_v2_clean = aws_gpu_pricing_v2_clean[
    [c for c in keep_cols if c in aws_gpu_pricing_v2_clean.columns]
].copy()

aws_gpu_summary_v2_clean = (
    aws_gpu_pricing_v2_clean
    .groupby("gpu_type")
    .agg(
        records=("instance_type", "count"),
        unique_instances=("instance_type", "nunique"),
        min_price=("hourly_price_usd", "min"),
        avg_price=("hourly_price_usd", "mean"),
        max_price=("hourly_price_usd", "max")
    )
    .reset_index()
    .sort_values("avg_price", ascending=False)
)

aws_gpu_pricing_v2_clean.to_csv(
    GPU_DIR / "aws_gpu_pricing_v2_clean.csv",
    index=False
)

aws_gpu_summary_v2_clean.to_csv(
    GPU_DIR / "aws_gpu_summary_v2_clean.csv",
    index=False
)

print("Saved:")
print(GPU_DIR / "aws_gpu_pricing_v2_clean.csv")
print(GPU_DIR / "aws_gpu_summary_v2_clean.csv")

print("\nPricing shape:", aws_gpu_pricing_v2_clean.shape)
print("Summary shape:", aws_gpu_summary_v2_clean.shape)

aws_gpu_summary_v2_clean

Saved:
C:\Users\user\Cloud Project\gpu_dataset_v2\aws_gpu_pricing_v2_clean.csv
C:\Users\user\Cloud Project\gpu_dataset_v2\aws_gpu_summary_v2_clean.csv

Pricing shape: (74, 11)
Summary shape: (10, 6)


,gpu_type,records,unique_instances,min_price,avg_price,max_price
2,H100,2,2,6.88000,30.960000,55.04000
9,V100,3,3,3.06000,13.260000,24.48000
1,Gaudi,1,1,13.10904,13.109040,13.10904
6,L4,32,8,0.80480,11.501563,85.35040
8,Trainium,2,2,1.34375,11.421875,21.50000
5,K80,3,3,0.90000,7.500000,14.40000
3,Inferentia,16,4,0.22800,5.846000,40.72100
4,Inferentia2,4,4,0.75820,5.549490,12.98127
0,A10G,8,8,1.00600,5.061250,16.28800
7,M60,3,3,1.14000,2.660000,4.56000


In [51]:
API_KEY = "YOUR_GCP_API_KEY"

In [52]:
import requests

url = "https://cloudbilling.googleapis.com/v1/services"

params = {
    "key": API_KEY
}

r = requests.get(url, params=params)

print(r.status_code)

data = r.json()
print(data.keys())

200
dict_keys(['services', 'nextPageToken'])


In [53]:
for s in data["services"]:
    if "Compute Engine" in s["displayName"]:
        print(s["name"])
        print(s["displayName"])

services/6F81-5844-456A
Compute Engine
services/9866-D980-657B
Google Click to Deploy Migrate for Compute Engine (formerly Velostrata)


In [54]:
import requests
import pandas as pd

service_id = "services/6F81-5844-456A"

all_skus = []
page_token = None

while True:

    url = f"https://cloudbilling.googleapis.com/v1/{service_id}/skus"

    params = {
        "key": API_KEY,
        "pageSize": 5000
    }

    if page_token:
        params["pageToken"] = page_token

    r = requests.get(url, params=params)

    data = r.json()

    all_skus.extend(data.get("skus", []))

    page_token = data.get("nextPageToken")

    print("Collected:", len(all_skus))

    if not page_token:
        break

print("\nFinal SKU count:", len(all_skus))

Collected: 5000
Collected: 10000
Collected: 15000
Collected: 20000
Collected: 25000
Collected: 30000
Collected: 31421

Final SKU count: 31421


In [55]:
print(all_skus[0].keys())

dict_keys(['name', 'skuId', 'description', 'category', 'serviceRegions', 'pricingInfo', 'serviceProviderName', 'geoTaxonomy'])


In [56]:
pd.Series([
    s["description"]
    for s in all_skus[:50]
]).head(20)

0     Sole Tenancy Premium for C4 Sole Tenancy Insta...
1     Spot Preemptible E2 Custom Instance Core runni...
2     Commitment v1: Memory-optimized Cpu in Phoenix...
3     Sole Tenancy Premium for C4 Sole Tenancy Insta...
4     M4Ultramem224 Sole Tenancy Instance Ram runnin...
5     Nvidia L4 GPU attached to Spot Preemptible VMs...
6         C3 Sole Tenancy Instance Ram running in Turin
7          N1 Predefined Instance Ram running in Zurich
8     Spot Preemptible Compute optimized Ram running...
9     Network Vpn Inter Region Data Transfer In from...
10    Licensing Fee for Windows Server 2012 BYOL (CP...
11      Commitment v1: C4A Arm Cpu in Dallas for 1 Year
12            Commitment v1: Cpu in Montreal for 1 Year
13    Hyperdisk Balanced IOPS Confidential Mode in Doha
14    Network Vpn Inter Region Data Transfer In from...
15         Sole Tenancy Instance RAM running in Jakarta
16    Hyperdisk Balanced Storage Pools Standard IOPS...
17    Sole Tenancy Premium for N2D AMD Sole Tena

In [57]:
gcp_skus = pd.DataFrame(all_skus)

print(gcp_skus.shape)
gcp_skus.head()

(31421, 8)


,name,skuId,description,category,serviceRegions,pricingInfo,serviceProviderName,geoTaxonomy
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,"{'serviceDisplayName': 'Compute Engine', 'reso...",[southamerica-west1],"[{'summary': '', 'pricingExpression': {'usageU...",Google,"{'type': 'REGIONAL', 'regions': ['southamerica..."
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,"{'serviceDisplayName': 'Compute Engine', 'reso...",[europe-west9],"[{'summary': '', 'pricingExpression': {'usageU...",Google,"{'type': 'REGIONAL', 'regions': ['europe-west9']}"
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,"{'serviceDisplayName': 'Compute Engine', 'reso...",[us-west8],"[{'summary': '', 'pricingExpression': {'usageU...",Google,"{'type': 'REGIONAL', 'regions': ['us-west8']}"
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,"{'serviceDisplayName': 'Compute Engine', 'reso...",[europe-north1],"[{'summary': '', 'pricingExpression': {'usageU...",Google,"{'type': 'REGIONAL', 'regions': ['europe-north..."
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,"{'serviceDisplayName': 'Compute Engine', 'reso...",[northamerica-northeast2],"[{'summary': '', 'pricingExpression': {'usageU...",Google,"{'type': 'REGIONAL', 'regions': ['northamerica..."


In [58]:
gcp_accelerator_keywords = [
    "Nvidia",
    "NVIDIA",
    "GPU",
    "A100",
    "H100",
    "H200",
    "L4",
    "T4",
    "V100",
    "P100",
    "P4",
    "RTX",
    "TPU",
    "Trillium"
]

pattern = "|".join(gcp_accelerator_keywords)

gcp_accel_raw = gcp_skus[
    gcp_skus["description"].str.contains(pattern, case=False, na=False)
].copy()

print(gcp_accel_raw.shape)

gcp_accel_raw[["description", "serviceRegions"]].head(50)

(2679, 8)


,description,serviceRegions
5,Nvidia L4 GPU attached to Spot Preemptible VMs...,[asia-east2]
23,Nvidia L4 GPU attached to Spot Preemptible VMs...,[northamerica-northeast1]
28,Nvidia Tesla P100 GPU running in Seoul,[asia-northeast3]
40,Commitment v1: H200 141GB GPU running in Nethe...,[europe-west4]
49,Nvidia H100 80GB Mega GPU running in Sydney,[australia-southeast1]
62,Licensing Fee for Ubuntu Pro FIPS 20.04 LTS (F...,[global]
82,Commitment v1: Nvidia Tesla A100 GPU running i...,[asia-east1]
109,Reserved Nvidia H100 80GB Mega GPU in Los Ange...,[us-west2]
123,Reserved Nvidia H100 80GB GPU in Frankfurt in ...,[europe-west3]
141,Commitment v1: Nvidia L4 GPU running in Berlin...,[europe-west10]


In [59]:
exclude_keywords = [
    "Commitment",
    "Spot",
    "Preemptible",
    "Reservation",
    "Sole Tenancy"
]

exclude_pattern = "|".join(exclude_keywords)

gcp_accel_clean = gcp_accel_raw[
    ~gcp_accel_raw["description"].str.contains(
        exclude_pattern,
        case=False,
        na=False
    )
].copy()

print(gcp_accel_clean.shape)

gcp_accel_clean[["description", "serviceRegions"]].head(100)

(1253, 8)


,description,serviceRegions
28,Nvidia Tesla P100 GPU running in Seoul,[asia-northeast3]
49,Nvidia H100 80GB Mega GPU running in Sydney,[australia-southeast1]
62,Licensing Fee for Ubuntu Pro FIPS 20.04 LTS (F...,[global]
109,Reserved Nvidia H100 80GB Mega GPU in Los Ange...,[us-west2]
123,Reserved Nvidia H100 80GB GPU in Frankfurt in ...,[europe-west3]
...,...,...
2301,Reserved Nvidia Tesla A100 GPU in Melbourne,[australia-southeast2]
2302,H200 141GB GPU attached to DWS Defined Duratio...,[me-west1]
2323,Licensing Fee for Ubuntu Pro FIPS Updates 26.0...,[global]
2326,Reserved Nvidia Tesla A100 80GB GPU in Dallas,[us-south1]


In [60]:
def detect_gcp_accelerator_type(text):
    text = str(text).upper()

    if "H200" in text:
        return "H200"
    elif "H100" in text:
        return "H100"
    elif "A100" in text:
        return "A100"
    elif "L4" in text:
        return "L4"
    elif "T4" in text:
        return "T4"
    elif "V100" in text:
        return "V100"
    elif "P100" in text:
        return "P100"
    elif "P4" in text:
        return "P4"
    elif "RTX" in text:
        return "RTX PRO 6000"
    elif "TPU V7" in text or "TRILLIUM" in text:
        return "TPU v7"
    elif "TPU V6E" in text:
        return "TPU v6e"
    elif "TPU V5P" in text:
        return "TPU v5p"
    elif "TPU V5E" in text:
        return "TPU v5e"
    elif "TPU" in text:
        return "TPU Other"
    else:
        return "Other"

gcp_accel_clean["gpu_type"] = gcp_accel_clean["description"].apply(
    detect_gcp_accelerator_type
)

gcp_accel_clean["gpu_type"].value_counts()

gpu_type
Other           369
TPU Other       202
A100            169
H100            163
H200             81
T4               81
L4               55
P4               37
P100             33
RTX PRO 6000     33
V100             30
Name: count, dtype: int64

In [61]:
valid_gcp_gpu_types = [
    "A100",
    "H100",
    "H200",
    "L4",
    "T4",
    "V100",
    "P100",
    "P4",
    "RTX PRO 6000"
]

gcp_gpu_only = gcp_accel_clean[
    gcp_accel_clean["gpu_type"].isin(valid_gcp_gpu_types)
].copy()

print(gcp_gpu_only.shape)

gcp_gpu_only[[
    "description",
    "gpu_type",
    "serviceRegions",
    "pricingInfo"
]].head(20)

(682, 9)


,description,gpu_type,serviceRegions,pricingInfo
28,Nvidia Tesla P100 GPU running in Seoul,P100,[asia-northeast3],"[{'summary': '', 'pricingExpression': {'usageU..."
49,Nvidia H100 80GB Mega GPU running in Sydney,H100,[australia-southeast1],"[{'summary': '', 'pricingExpression': {'usageU..."
109,Reserved Nvidia H100 80GB Mega GPU in Los Ange...,H100,[us-west2],"[{'summary': '', 'pricingExpression': {'usageU..."
123,Reserved Nvidia H100 80GB GPU in Frankfurt in ...,H100,[europe-west3],"[{'summary': '', 'pricingExpression': {'usageU..."
155,Nvidia L4 GPU running in Singapore,L4,[asia-southeast1],"[{'summary': '', 'pricingExpression': {'usageU..."
189,Reserved Nvidia H100 80GB GPU in Singapore in ...,H100,[asia-southeast1],"[{'summary': '', 'pricingExpression': {'usageU..."
192,Nvidia Tesla T4 GPU attached to DWS Defined Du...,T4,[asia-southeast3],"[{'summary': '', 'pricingExpression': {'usageU..."
278,Nvidia Tesla T4 GPU running in Japan,T4,[asia-northeast1],"[{'summary': '', 'pricingExpression': {'usageU..."
365,Nvidia Tesla T4 GPU attached to DWS Defined Du...,T4,[us-west4],"[{'summary': '', 'pricingExpression': {'usageU..."
382,RTX 6000 96GB running in Jakarta,RTX PRO 6000,[asia-southeast2],"[{'summary': '', 'pricingExpression': {'usageU..."


In [62]:
def extract_gcp_hourly_price(pricing_info):
    try:
        if not pricing_info:
            return None

        pricing = pricing_info[0]
        rate = pricing["pricingExpression"]["tieredRates"][0]

        unit_price = rate["unitPrice"]

        units = float(unit_price.get("units", 0))
        nanos = float(unit_price.get("nanos", 0)) / 1e9

        return units + nanos

    except Exception:
        return None


gcp_gpu_only["hourly_price_usd"] = gcp_gpu_only["pricingInfo"].apply(
    extract_gcp_hourly_price
)

gcp_gpu_only[[
    "description",
    "gpu_type",
    "hourly_price_usd",
    "serviceRegions"
]].head(50)

,description,gpu_type,hourly_price_usd,serviceRegions
28,Nvidia Tesla P100 GPU running in Seoul,P100,1.600000,[asia-northeast3]
49,Nvidia H100 80GB Mega GPU running in Sydney,H100,12.930340,[australia-southeast1]
109,Reserved Nvidia H100 80GB Mega GPU in Los Ange...,H100,4.841121,[us-west2]
123,Reserved Nvidia H100 80GB GPU in Frankfurt in ...,H100,4.570091,[europe-west3]
155,Nvidia L4 GPU running in Singapore,L4,0.690922,[asia-southeast1]
189,Reserved Nvidia H100 80GB GPU in Singapore in ...,H100,4.570091,[asia-southeast1]
192,Nvidia Tesla T4 GPU attached to DWS Defined Du...,T4,0.420000,[asia-southeast3]
278,Nvidia Tesla T4 GPU running in Japan,T4,0.370000,[asia-northeast1]
365,Nvidia Tesla T4 GPU attached to DWS Defined Du...,T4,0.370000,[us-west4]
382,RTX 6000 96GB running in Jakarta,RTX PRO 6000,1.473321,[asia-southeast2]


In [63]:
gcp_gpu_only["hourly_price_usd"].isna().sum(), len(gcp_gpu_only)

(np.int64(0), 682)

In [64]:
gcp_gpu_summary_check = (
    gcp_gpu_only
    .groupby("gpu_type")
    .agg(
        records=("description", "count"),
        min_price=("hourly_price_usd", "min"),
        avg_price=("hourly_price_usd", "mean"),
        max_price=("hourly_price_usd", "max")
    )
    .reset_index()
    .sort_values("avg_price", ascending=False)
)

gcp_gpu_summary_check

,gpu_type,records,min_price,avg_price,max_price
1,H100,163,4.200761,6.848646,15.674480
2,H200,81,0.000000,5.678299,11.174090
8,V100,30,0.000000,2.336261,3.968000
0,A100,169,0.000000,2.063868,6.284928
4,P100,33,1.460000,1.707327,2.336000
6,RTX PRO 6000,33,0.547830,0.967650,1.473321
5,P4,37,0.600000,0.695141,0.960000
3,L4,55,0.558206,0.675517,0.896064
7,T4,81,0.350000,0.409825,0.560000


In [65]:
gcp_gpu_only = gcp_gpu_only[
    gcp_gpu_only["hourly_price_usd"] > 0
].copy()

In [66]:
gcp_gpu_summary_check = (
    gcp_gpu_only
    .groupby("gpu_type")
    .agg(
        records=("description", "count"),
        min_price=("hourly_price_usd", "min"),
        avg_price=("hourly_price_usd", "mean"),
        max_price=("hourly_price_usd", "max")
    )
    .reset_index()
    .sort_values("avg_price", ascending=False)
)

gcp_gpu_summary_check

,gpu_type,records,min_price,avg_price,max_price
1,H100,163,4.200761,6.848646,15.674480
2,H200,80,4.575323,5.749278,11.174090
0,A100,92,1.613649,3.791236,6.284928
8,V100,24,2.480000,2.920327,3.968000
4,P100,33,1.460000,1.707327,2.336000
6,RTX PRO 6000,33,0.547830,0.967650,1.473321
5,P4,37,0.600000,0.695141,0.960000
3,L4,55,0.558206,0.675517,0.896064
7,T4,81,0.350000,0.409825,0.560000


In [67]:
gcp_gpu_only.to_csv(
    r"C:\Users\user\Cloud Project\gpu_dataset_v2\gcp_gpu_pricing_v2_clean.csv",
    index=False
)

gcp_gpu_summary_check.to_csv(
    r"C:\Users\user\Cloud Project\gpu_dataset_v2\gcp_gpu_summary_v2_clean.csv",
    index=False
)

In [68]:
aws_gpu_final["Provider"] = "AWS"
azure_gpu_v2["Provider"] = "Azure"
gcp_gpu_only["Provider"] = "GCP"

In [70]:
aws_gpu_final.columns

Index(['sku', 'productFamily', 'servicecode', 'location', 'locationType',
       'instanceType', 'currentGeneration', 'instanceFamily', 'vcpu',
       'physicalProcessor', 'clockSpeed', 'memory', 'storage',
       'networkPerformance', 'processorArchitecture', 'tenancy',
       'operatingSystem', 'licenseModel', 'usagetype', 'operation',
       'availabilityzone', 'capacitystatus', 'classicnetworkingsupport',
       'dedicatedEbsThroughput', 'dedicatedEbsThroughputDescription', 'ecu',
       'enhancedNetworkingSupported', 'gpuMemory', 'instanceFamilyCategory',
       'instancesku', 'intelAvxAvailable', 'intelAvx2Available',
       'intelTurboAvailable', 'marketoption', 'normalizationSizeFactor',
       'preInstalledSw', 'regionCode', 'servicename', 'vpcnetworkingsupport',
       'processorFeatures', 'gpu', 'instanceCapacity-12xlarge',
       'instanceCapacity-16xlarge', 'instanceCapacity-24xlarge',
       'instanceCapacity-2xlarge', 'instanceCapacity-32xlarge',
       'instanceCapacity

In [71]:
aws_final = aws_gpu_final[
    ["Provider", "gpu_type", "instanceType", "hourly_price_usd"]
].copy()

aws_final = aws_final.rename(columns={
    "Provider": "provider",
    "instanceType": "instance_type"
})

aws_final.head()

,provider,gpu_type,instance_type,hourly_price_usd
0,AWS,L4,g6.24xlarge,6.6752
1,AWS,L4,g6.4xlarge,3.2432
2,AWS,A10G,g5.8xlarge,2.4480
3,AWS,Inferentia,inf1.6xlarge,4.0600
4,AWS,Inferentia,inf1.24xlarge,4.7210


In [72]:
print(azure_gpu_v2_final.columns)
print(gcp_gpu_only.columns)

Index(['region', 'product_name', 'instance_type', 'meter_name', 'gpu_type',
       'hourly_price_usd', 'currency', 'unitOfMeasure', 'provider',
       'collection_date'],
      dtype='object')
Index(['name', 'skuId', 'description', 'category', 'serviceRegions',
       'pricingInfo', 'serviceProviderName', 'geoTaxonomy', 'gpu_type',
       'hourly_price_usd', 'Provider'],
      dtype='object')


In [73]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(r"C:\Users\user\Cloud Project\gpu_dataset_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# AWS
aws_final = aws_gpu_final[
    ["Provider", "gpu_type", "instanceType", "hourly_price_usd"]
].copy()

aws_final = aws_final.rename(columns={
    "Provider": "provider",
    "instanceType": "instance_type"
})

aws_final["source_table"] = "aws_gpu_pricing_v2_clean"

# Azure
azure_final = azure_gpu_v2_final[
    ["provider", "gpu_type", "instance_type", "hourly_price_usd"]
].copy()

azure_final["source_table"] = "azure_gpu_pricing_v2_clean"

# GCP
gcp_final = gcp_gpu_only[
    ["Provider", "gpu_type", "description", "hourly_price_usd"]
].copy()

gcp_final = gcp_final.rename(columns={
    "Provider": "provider",
    "description": "instance_type"
})

gcp_final["source_table"] = "gcp_gpu_pricing_v2_clean"

# 合併
accelerator_master = pd.concat(
    [aws_final, azure_final, gcp_final],
    ignore_index=True
)

# 清理
accelerator_master["provider"] = accelerator_master["provider"].str.strip()
accelerator_master["gpu_type"] = accelerator_master["gpu_type"].str.strip()
accelerator_master["instance_type"] = accelerator_master["instance_type"].astype(str).str.strip()

accelerator_master = accelerator_master[
    accelerator_master["hourly_price_usd"].notna() &
    (accelerator_master["hourly_price_usd"] > 0)
].copy()

# 分類：GPU / AI Accelerator / TPU
def accelerator_category(gpu_type):
    gpu_type = str(gpu_type)

    if gpu_type in ["Inferentia", "Inferentia2"]:
        return "AWS Inference Chip"
    elif gpu_type in ["Trainium", "Trainium2"]:
        return "AWS Training Chip"
    elif gpu_type == "Gaudi":
        return "Intel AI Accelerator"
    elif "TPU" in gpu_type:
        return "Google TPU"
    else:
        return "NVIDIA GPU"

accelerator_master["accelerator_category"] = accelerator_master["gpu_type"].apply(
    accelerator_category
)

# 輸出 Master
accelerator_master.to_csv(
    OUTPUT_DIR / "accelerator_master_v2.csv",
    index=False
)

# Summary 1: provider-level
accelerator_provider_summary = (
    accelerator_master
    .groupby("provider")
    .agg(
        records=("instance_type", "count"),
        unique_accelerators=("gpu_type", "nunique"),
        unique_instances=("instance_type", "nunique"),
        avg_price=("hourly_price_usd", "mean"),
        min_price=("hourly_price_usd", "min"),
        max_price=("hourly_price_usd", "max")
    )
    .reset_index()
)

accelerator_provider_summary.to_csv(
    OUTPUT_DIR / "accelerator_provider_summary_v2.csv",
    index=False
)

# Summary 2: provider × accelerator type
accelerator_type_summary = (
    accelerator_master
    .groupby(["provider", "gpu_type", "accelerator_category"])
    .agg(
        records=("instance_type", "count"),
        unique_instances=("instance_type", "nunique"),
        avg_price=("hourly_price_usd", "mean"),
        min_price=("hourly_price_usd", "min"),
        max_price=("hourly_price_usd", "max")
    )
    .reset_index()
    .sort_values(["provider", "avg_price"], ascending=[True, False])
)

accelerator_type_summary.to_csv(
    OUTPUT_DIR / "accelerator_type_summary_v2.csv",
    index=False
)

print("Saved:")
print(OUTPUT_DIR / "accelerator_master_v2.csv")
print(OUTPUT_DIR / "accelerator_provider_summary_v2.csv")
print(OUTPUT_DIR / "accelerator_type_summary_v2.csv")

print("\nMaster shape:", accelerator_master.shape)
print("\nProvider summary:")
display(accelerator_provider_summary)

print("\nType summary:")
display(accelerator_type_summary.head(30))

Saved:
C:\Users\user\Cloud Project\gpu_dataset_v2\accelerator_master_v2.csv
C:\Users\user\Cloud Project\gpu_dataset_v2\accelerator_provider_summary_v2.csv
C:\Users\user\Cloud Project\gpu_dataset_v2\accelerator_type_summary_v2.csv

Master shape: (724, 6)

Provider summary:


,provider,records,unique_accelerators,unique_instances,avg_price,min_price,max_price
0,AWS,74,10,38,9.356848,0.228,85.35040
1,Azure,52,4,27,27.670596,0.454,102.73600
2,GCP,598,9,598,3.644643,0.350,15.67448



Type summary:


,provider,gpu_type,accelerator_category,records,unique_instances,avg_price,min_price,max_price
2,AWS,H100,NVIDIA GPU,2,2,30.960000,6.880000,55.040000
9,AWS,V100,NVIDIA GPU,3,3,13.260000,3.060000,24.480000
1,AWS,Gaudi,Intel AI Accelerator,1,1,13.109040,13.109040,13.109040
6,AWS,L4,NVIDIA GPU,32,8,11.501563,0.804800,85.350400
8,AWS,Trainium,AWS Training Chip,2,2,11.421875,1.343750,21.500000
5,AWS,K80,NVIDIA GPU,3,3,7.500000,0.900000,14.400000
3,AWS,Inferentia,AWS Inference Chip,16,4,5.846000,0.228000,40.721000
4,AWS,Inferentia2,AWS Inference Chip,4,4,5.549490,0.758200,12.981270
0,AWS,A10G,NVIDIA GPU,8,8,5.061250,1.006000,16.288000
7,AWS,M60,NVIDIA GPU,3,3,2.660000,1.140000,4.560000


In [74]:
gcp_accel_clean[
    gcp_accel_clean["gpu_type"] == "TPU Other"
][["description"]].drop_duplicates().head(100)

,description
340,Reserved TpuV5p in Delhi in Calendar Mode
478,Reserved TpuV5p in Santiago in Calendar Mode
563,Reserved TpuV5p in Columbus in Calendar Mode
643,TpuV5e running in Delhi
1059,TpuV5p running in Sao Paulo
...,...
14133,Capacity Optimized TpuV6e running in Singapore
14190,Reserved TpuV5p in Dammam in Calendar Mode
14204,Reserved V5e TPU in Frankfurt in Calendar Mode
14341,TpuV6e running in Virginia


In [75]:
tpu_df = gcp_accel_clean[
    gcp_accel_clean["gpu_type"] == "TPU Other"
].copy()

tpu_df["description"].value_counts().head(50)

description
Reserved TpuV5p in Delhi in Calendar Mode                1
Reserved TpuV5p in Santiago in Calendar Mode             1
Reserved TpuV5p in Columbus in Calendar Mode             1
TpuV5e running in Delhi                                  1
TpuV5p running in Sao Paulo                              1
TpuV5e running in London                                 1
Reserved TpuV5p in Zurich in Calendar Mode               1
Reserved TpuV5p in Oklahoma in Calendar Mode             1
Reserved TpuV5p in Netherlands in Calendar Mode          1
Reserved V5e TPU in Sao Paulo in Calendar Mode           1
TPU7x running in Americas                                1
TpuV5e running in Melbourne                              1
Reserved TpuV5p in Milan in Calendar Mode                1
Reserved TpuV5p in Doha in Calendar Mode                 1
TPU7x running in London                                  1
TpuV5p running in Phoenix                                1
Reserved V5e TPU in South Carolina in Calend

In [76]:
tpu_df = gcp_accel_clean[
    gcp_accel_clean["gpu_type"] == "TPU Other"
].copy()

def detect_tpu_type(text):

    text = str(text).upper()

    if "TPUV5E" in text or "V5E TPU" in text:
        return "TPU v5e"

    elif "TPUV5P" in text or "V5P TPU" in text:
        return "TPU v5p"

    elif "TPUV6E" in text:
        return "TPU v6e"

    elif "TRILLIUM" in text:
        return "TPU v7"

    else:
        return "TPU Other"

tpu_df["gpu_type"] = tpu_df["description"].apply(
    detect_tpu_type
)

tpu_df["gpu_type"].value_counts()

gpu_type
TPU v5p      81
TPU v5e      81
TPU Other    20
TPU v6e      20
Name: count, dtype: int64

In [77]:
tpu_df["hourly_price_usd"] = tpu_df["pricingInfo"].apply(
    extract_gcp_hourly_price
)

tpu_df = tpu_df[
    tpu_df["hourly_price_usd"] > 0
].copy()

In [78]:
tpu_df = tpu_df[
    tpu_df["gpu_type"] != "TPU Other"
].copy()

In [79]:
tpu_summary = (
    tpu_df
    .groupby("gpu_type")
    .agg(
        records=("description","count"),
        min_price=("hourly_price_usd","min"),
        avg_price=("hourly_price_usd","mean"),
        max_price=("hourly_price_usd","max")
    )
    .reset_index()
    .sort_values("avg_price", ascending=False)
)

tpu_summary

,gpu_type,records,min_price,avg_price,max_price
1,TPU v5p,81,2.94,3.904248,6.48
2,TPU v6e,20,2.70,2.916000,3.24
0,TPU v5e,81,0.84,1.142400,1.92


In [80]:
gcp_tpu_final = tpu_df[
    ["gpu_type", "description", "hourly_price_usd"]
].copy()

gcp_tpu_final["provider"] = "GCP"

gcp_tpu_final = gcp_tpu_final.rename(columns={
    "description": "instance_type"
})

gcp_tpu_final["accelerator_category"] = "Google TPU"

In [81]:
accelerator_master_v3 = pd.concat(
    [
        accelerator_master,
        gcp_tpu_final
    ],
    ignore_index=True
)

In [82]:
accelerator_master_v3 = pd.concat(
    [
        accelerator_master,
        gcp_tpu_final
    ],
    ignore_index=True
)

In [83]:
tpu_df = tpu_df[
    tpu_df["hourly_price_usd"] > 0
].copy()

In [84]:
tpu_df = tpu_df[
    tpu_df["gpu_type"] != "TPU Other"
].copy()

In [85]:
tpu_summary = (
    tpu_df
    .groupby("gpu_type")
    .agg(
        records=("description", "count"),
        min_price=("hourly_price_usd", "min"),
        avg_price=("hourly_price_usd", "mean"),
        max_price=("hourly_price_usd", "max")
    )
    .reset_index()
    .sort_values("avg_price", ascending=False)
)

tpu_summary

,gpu_type,records,min_price,avg_price,max_price
1,TPU v5p,81,2.94,3.904248,6.48
2,TPU v6e,20,2.70,2.916000,3.24
0,TPU v5e,81,0.84,1.142400,1.92


In [86]:
gcp_tpu_final = tpu_df[
    ["gpu_type", "description", "hourly_price_usd"]
].copy()

gcp_tpu_final["provider"] = "GCP"

gcp_tpu_final = gcp_tpu_final.rename(
    columns={
        "description": "instance_type"
    }
)

gcp_tpu_final["accelerator_category"] = "Google TPU"

In [87]:
accelerator_master_v3 = pd.concat(
    [
        accelerator_master,
        gcp_tpu_final
    ],
    ignore_index=True
)

In [88]:
accelerator_master_v3.to_csv(
    r"C:\Users\user\Cloud Project\gpu_dataset_v2\accelerator_master_v3.csv",
    index=False
)

In [90]:
for var in dir():
    obj = globals()[var]

    try:
        if hasattr(obj, "shape"):
            print(var, obj.shape)
    except:
        pass

_ (3, 5)
_13 (50, 4)
_15 (30, 3)
_20 (5, 21)
_22 (50, 10)
_23 (4, 6)
_3 (5, 21)
_30 (3,)
_32 (5, 79)
_33 (20, 8)
_34 (50, 8)
_35 (11,)
_36 (20, 6)
_37 (50, 10)
_39 (11, 6)
_4 (50, 7)
_43 (10, 6)
_49 (10, 6)
_5 (50, 6)
_56 (20,)
_57 (5, 8)
_58 (50, 2)
_59 (100, 2)
_6 (7, 6)
_60 (11,)
_61 (20, 4)
_62 (50, 4)
_64 (9, 5)
_66 (9, 5)
_7 (50, 4)
_70 (86,)
_71 (5, 4)
_74 (100, 1)
_75 (50,)
_76 (4,)
_79 (3, 5)
_8 (30,)
_85 (3, 5)
_9 ()
__ (3, 5)
___ (4,)
accelerator_master (724, 6)
accelerator_master_v3 (906, 6)
accelerator_provider_summary (3, 7)
accelerator_type_summary (23, 8)
attrs (2006559, 77)
aws_ec2_us_east (3817, 79)
aws_final (74, 5)
aws_gpu_candidates (79, 79)
aws_gpu_final (74, 86)
aws_gpu_pricing (79, 85)
aws_gpu_pricing_v2_clean (74, 11)
aws_gpu_summary_v2_clean (10, 6)
aws_price_df (79, 6)
aws_products (2006559, 79)
azure_final (52, 5)
azure_gpu (5527, 23)
azure_gpu_clean (54, 23)
azure_gpu_summary_check (4, 6)
azure_gpu_v2 (5527, 11)
azure_gpu_v2_final (52, 10)
azure_raw (8537, 